In [1]:
!pip install -q transformers datasets sacrebleu sentencepiece \
             accelerate evaluate

In [2]:
!pip show transformers

Name: transformers
Version: 5.4.0
Summary: Transformers: the model-definition framework for state-of-the-art machine learning models in text, vision, audio, and multimodal models, for both inference and training.
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages
Requires: huggingface-hub, numpy, packaging, pyyaml, regex, safetensors, tokenizers, tqdm, typer
Required-by: 


In [3]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════

MODEL_ID  = "facebook/nllb-200-distilled-600M"
SRC_LANG  = "twi_Latn"   # Twi (Akan, Latin script)
TGT_LANG  = "eng_Latn"   # English
MAX_LEN   = 128          # Reduced from 256 for memory

# Training - reduced batch for 16GB VRAM
NUM_EPOCHS       = 10
BATCH_SIZE       = 4      # Small batch for memory
GRAD_ACCUM       = 8      # Effective batch = 32
LEARNING_RATE    = 3e-5
WARMUP_RATIO     = 0.06
WEIGHT_DECAY     = 0.01
LABEL_SMOOTHING  = 0.1

# Paths
DATA_DIR        = "/teamspace/studios/this_studio/"
TRAIN_CSV       = f"{DATA_DIR}train_twi_to_en.csv"
VAL_CSV         = f"{DATA_DIR}val_twi_to_en.csv"
TEST_CSV        = f"{DATA_DIR}test_twi_to_en.csv"

OUTPUT_DIR      = "./twi-en-nllb-v2"
HF_REPO         = "ninte/twi-en-nllb-v2"

print("Configuration loaded.")
print(f"  Model     : {MODEL_ID}")
print(f"  Direction : {SRC_LANG} → {TGT_LANG}")
print(f"  Max length: {MAX_LEN}")
print(f"  Batch     : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")

Configuration loaded.
  Model     : facebook/nllb-200-distilled-600M
  Direction : twi_Latn → eng_Latn
  Max length: 128
  Batch     : 4 (effective: 32)


In [4]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Sanity check columns
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert 'twi' in df.columns and 'english' in df.columns, \
        f"{name}: expected 'twi' and 'english' columns, got {list(df.columns)}"
    assert df[['twi','english']].isna().sum().sum() == 0, \
        f"{name}: unexpected nulls found"

print(f"Train : {len(train_df):,} pairs")
print(f"Val   : {len(val_df):,} pairs")
print(f"Test  : {len(test_df):,} pairs")
print(f"Total : {len(train_df)+len(val_df)+len(test_df):,} pairs")
print("\nSample:")
train_df.sample(3, random_state=42)

Train : 7,732 pairs
Val   : 967 pairs
Test  : 967 pairs
Total : 9,666 pairs

Sample:


,twi,english
1720,Bɔ mmɔden sɛ wonam wo kasa so bɛhyɛ nkuran na ...,Always try to encourage and save lives through...
748,"Ɛnnɛ deɛ, abɛɛfo kuayɔ nti, akuafoɔ dodoɔ no a...","Today, thanks to modern farming, most farmers ..."
7074,"Ɛno nti, wɔatumi ahuri adi kan a wɔne obiara m...","And that, they have been able to win and secur..."


In [5]:
!ls

Untitled.ipynb	      test_combined.csv    train_twi_to_en.csv
combined_cleaned.csv  test_en_to_twi.csv   twi-en-nllb-v2
data_statistics.png   test_twi_to_en.csv   val_combined.csv
main.py		      train_combined.csv   val_en_to_twi.csv
nllb_v2.ipynb	      train_en_to_twi.csv  val_twi_to_en.csv


In [6]:
!pwd

/teamspace/studios/this_studio


In [7]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print(f"Loading {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Enable gradient checkpointing to save VRAM during training
model.gradient_checkpointing_enable()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Device      : {device}")
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"Gradient checkpointing: enabled")
if torch.cuda.is_available():
    print(f"VRAM used   : {torch.cuda.memory_allocated()/1e9:.2f} GB")

Loading facebook/nllb-200-distilled-600M...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Device      : cuda
Model params: 615.1M
Gradient checkpointing: enabled
VRAM used   : 2.47 GB


In [8]:
from datasets import Dataset

def tokenize_batch(batch):
    """Tokenize for NLLB Twi → English direction with explicit decoder_input_ids."""
    tokenizer.src_lang = SRC_LANG  # twi_Latn

    # Encode source (Twi)
    model_inputs = tokenizer(
        batch["twi"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    
    # Encode target (English) using text_target parameter
    labels = tokenizer(
        text_target=batch["english"],
        max_length=MAX_LEN,
        truncation=True,
        padding="max_length",
    )
    
    # Create decoder_input_ids by shifting labels right
    # Prepend language token (eng_Latn) instead of just pad
    eng_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    decoder_input_ids = []
    for label_ids in labels["input_ids"]:
        # Shift right: prepend target language token, remove last token
        shifted = [eng_token_id] + label_ids[:-1]
        decoder_input_ids.append(shifted)
    
    model_inputs["decoder_input_ids"] = decoder_input_ids
    model_inputs["decoder_attention_mask"] = labels["attention_mask"]
    
    # Labels: mask padding tokens with -100
    model_inputs["labels"] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels["input_ids"]
    ]
    
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['twi','english']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['twi','english']].reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df[['twi','english']].reset_index(drop=True))

print("Tokenizing with explicit decoder_input_ids...")
train_tok = train_ds.map(tokenize_batch, batched=True, batch_size=256,
                          remove_columns=['twi','english'])
val_tok   = val_ds.map(tokenize_batch,   batched=True, batch_size=256,
                          remove_columns=['twi','english'])
test_tok  = test_ds.map(tokenize_batch,  batched=True, batch_size=256,
                          remove_columns=['twi','english'])

print("Tokenization complete.")
print(f"Sample keys: {list(train_tok[0].keys())}")
print(f"Train size: {len(train_tok)}, Val size: {len(val_tok)}, Test size: {len(test_tok)}")

Tokenizing with explicit decoder_input_ids...


Map:   0%|          | 0/7732 [00:00<?, ? examples/s]

Map:   0%|          | 0/967 [00:00<?, ? examples/s]

Map:   0%|          | 0/967 [00:00<?, ? examples/s]

Tokenization complete.
Sample keys: ['input_ids', 'attention_mask', 'decoder_input_ids', 'decoder_attention_mask', 'labels']
Train size: 7732, Val size: 967, Test size: 967


In [9]:
import evaluate
import numpy as np

bleu_metric = evaluate.load("sacrebleu")
chrf_metric = evaluate.load("chrf")
ter_metric  = evaluate.load("ter")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.argmax(preds, axis=-1) if preds.ndim == 3 else preds
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    bleu = bleu_metric.compute(predictions=decoded_preds,
                                references=[[r] for r in decoded_labels])
    chrf = chrf_metric.compute(predictions=decoded_preds,
                                references=decoded_labels)
    ter  = ter_metric.compute(predictions=decoded_preds,
                               references=[[r] for r in decoded_labels])
    return {
        "bleu": round(bleu["score"], 2),
        "chrf": round(chrf["score"], 2),
        "ter":  round(ter["score"],  2),  # lower is better
    }

print("Metrics ready: BLEU, chrF, TER")

Metrics ready: BLEU, chrF, TER


In [10]:
from transformers import Seq2SeqTrainingArguments
import torch
import os

# ═══════════════════════════════════════════════════════════════
# Custom collator for pre-tokenized data with decoder_input_ids
# ═══════════════════════════════════════════════════════════════
def custom_collator(features):
    """Stack pre-padded tensors into batches."""
    batch = {}
    for key in features[0].keys():
        batch[key] = torch.tensor([f[key] for f in features])
    return batch

data_collator = custom_collator

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="chrf",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,
    report_to="none",
    label_smoothing_factor=LABEL_SMOOTHING,
)

print(f"Training args ready — {len(train_df):,} training pairs")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training args ready — 7,732 training pairs
Effective batch: 32


In [11]:
from transformers import Seq2SeqTrainer
import os

# ═══════════════════════════════════════════════════════════════
# Custom Trainer - filters problematic keys for NLLB
# ═══════════════════════════════════════════════════════════════
class NLLBTrainer(Seq2SeqTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs.pop("decoder_inputs_embeds", None)
        inputs.pop("cache_position", None)
        return super().compute_loss(model, inputs, return_outputs=return_outputs, **kwargs)
    
    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs.pop("decoder_inputs_embeds", None)
        inputs.pop("cache_position", None)
        return super().prediction_step(model, inputs, prediction_loss_only, ignore_keys=ignore_keys)

trainer = NLLBTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ── Checkpoint resume ──
checkpoints = []
if os.path.exists(OUTPUT_DIR):
    checkpoints = [f for f in os.listdir(OUTPUT_DIR) if f.startswith("checkpoint-")]

if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from = os.path.join(OUTPUT_DIR, latest)
    print(f"Resuming from: {resume_from}")
else:
    resume_from = None
    print("Starting fresh training run")

trainer.train(resume_from_checkpoint=resume_from)

Resuming from: ./twi-en-nllb-v2/checkpoint-2420


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss


TrainOutput(global_step=2420, training_loss=0.0, metrics={'train_runtime': 0.0051, 'train_samples_per_second': 15307447.62, 'train_steps_per_second': 479100.145, 'total_flos': 2.094506597941248e+16, 'train_loss': 0.0, 'epoch': 10.0})

In [12]:
import sacrebleu as sb

def full_evaluate(model, tokenizer, df, src_col='twi', ref_col='english',
                   src_lang=SRC_LANG, tgt_lang=TGT_LANG, label=''):
    model.eval()
    tgt_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
    preds, refs  = [], []

    for _, row in df.iterrows():
        tokenizer.src_lang = src_lang
        inputs = tokenizer(row[src_col], return_tensors="pt",
                           truncation=True, max_length=MAX_LEN)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(
                **inputs,
                forced_bos_token_id=tgt_token_id,
                num_beams=4, max_length=MAX_LEN,
                no_repeat_ngram_size=3, length_penalty=1.0, early_stopping=True,
            )
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row[ref_col])

    bleu = sb.corpus_bleu(preds, [refs])
    chrf = sb.corpus_chrf(preds, [refs])
    ter  = sb.corpus_ter(preds, [refs])

    print(f"\n{'='*55}")
    print(f"  {label} ({len(df):,} pairs)")
    print(f"{'='*55}")
    print(f"  BLEU   : {bleu.score:.2f}")
    print(f"  chrF   : {chrf.score:.2f}")
    print(f"  TER    : {ter.score:.2f}  (lower is better)")
    print(f"  BLEU 1g: {bleu.precisions[0]:.2f}")
    print(f"  BLEU 4g: {bleu.precisions[3]:.2f}")
    print(f"  BP     : {bleu.bp:.4f}")
    print(f"{'='*55}\n")
    return {"bleu": bleu.score, "chrf": chrf.score, "ter": ter.score,
            "preds": preds, "refs": refs}

results = full_evaluate(model, tokenizer, test_df, label="Twi → English Test Set")

# Baseline comparison
print("  v1 MarianMT baseline:")
print("  Twi→En  BLEU 10.50 | chrF 32.30 | TER 87.64")
delta_bleu = results['bleu'] - 10.50
delta_chrf = results['chrf'] - 32.30
print(f"  Improvement: BLEU {delta_bleu:+.2f} | chrF {delta_chrf:+.2f}")


  Twi → English Test Set (967 pairs)
  BLEU   : 27.30
  chrF   : 51.03
  TER    : 61.48  (lower is better)
  BLEU 1g: 57.11
  BLEU 4g: 14.17
  BP     : 1.0000

  v1 MarianMT baseline:
  Twi→En  BLEU 10.50 | chrF 32.30 | TER 87.64
  Improvement: BLEU +16.80 | chrF +18.73


In [13]:
def translate(text, src_lang=SRC_LANG, tgt_lang=TGT_LANG):
    tokenizer.src_lang = src_lang
    tgt_id  = tokenizer.convert_tokens_to_ids(tgt_lang)
    inputs  = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    inputs  = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, forced_bos_token_id=tgt_id,
                              num_beams=4, max_length=MAX_LEN,
                              no_repeat_ngram_size=3, length_penalty=1.0,
                              early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Sample from actual test set
print("=" * 70)
print("TRANSLATION EXAMPLES — Twi → English")
print("=" * 70)
for _, row in test_df.sample(5, random_state=1).iterrows():
    print(f"\nTwi    : {row['twi']}")
    print(f"Pred   : {translate(row['twi'])}")
    print(f"Ref    : {row['english']}")

TRANSLATION EXAMPLES — Twi → English

Twi    : Osukɛseɛ bi a asasewosoɔ ne agradaa di akyire tɔeɛ maa deɛ na wɔhyia no bu guu mu.
Pred   : A massive earthquake and tsunami followed, causing the encounter to collapse.
Ref    : A heavy rain with earthquake and thunder fell and collapsed their place of gathering.

Twi    : Mpanimfoɔ a wɔda mansini no ano no ne nnipa a wɔbubu nnua no ayɛ baako sɛdeɛ ɛbɛyɛ a wɔnɛnya sika afiri wɔn hɔ
Pred   : The elders at the border of the district have teamed up with the loggers to get money from them.
Ref    : The district officials connive with loggers to make money out of trees.

Twi    : Ɛsɛ sɛ mmofra di aduan pa na ama wɔanyin yiye.
Pred   : Children need to eat healthy foods to grow well.
Ref    : Children need to feed well in order to grow well.

Twi    : Ahennie ahodoɔ a ɛwɔ ɔman no mu no nso bɛtumi anya mmoa afiri aban no hɔ.
Pred   : Other governments in the country can also get support from the government.
Ref    : The kingdoms in the country c

In [16]:
from huggingface_hub import notebook_login
notebook_login()

In [17]:
model.config.update({
    "src_lang": SRC_LANG,
    "tgt_lang": TGT_LANG,
    "language": ["tw", "en"],
    "tags": ["translation", "twi", "akan", "ghana", "low-resource", "nllb"],
    "license": "cc-by-nc-4.0",
})

print(f"Pushing to {HF_REPO}...")
model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"\nPublished: https://huggingface.co/{HF_REPO}")

Pushing to ninte/twi-en-nllb-v2...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Published: https://huggingface.co/ninte/twi-en-nllb-v2


In [18]:
print("\n" + "="*70)
print("TRAINING SUMMARY — Twi → English")
print("="*70)
print(f"  Base model     : {MODEL_ID}")
print(f"  Direction      : {SRC_LANG} → {TGT_LANG}")
print(f"  Train pairs    : {len(train_df):,}")
print(f"  Val pairs      : {len(val_df):,}")
print(f"  Test pairs     : {len(test_df):,}")
print(f"  Epochs         : {NUM_EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE} (effective: {BATCH_SIZE * GRAD_ACCUM})")
print(f"  Learning rate  : {LEARNING_RATE}")
print(f"  Max seq length : {MAX_LEN}")
print(f"  Label smoothing: {LABEL_SMOOTHING}")
print(f"  Best metric    : chrF (primary for Twi)")
print(f"\n  Test BLEU      : {results['bleu']:.2f}  (v1 baseline: 10.50)")
print(f"  Test chrF      : {results['chrf']:.2f}  (v1 baseline: 32.30)")
print(f"  Test TER       : {results['ter']:.2f}  (v1 baseline: 87.64)")
print(f"\n  Published      : https://huggingface.co/{HF_REPO}")
print("="*70)


TRAINING SUMMARY — Twi → English
  Base model     : facebook/nllb-200-distilled-600M
  Direction      : twi_Latn → eng_Latn
  Train pairs    : 7,732
  Val pairs      : 967
  Test pairs     : 967
  Epochs         : 10
  Batch size     : 4 (effective: 32)
  Learning rate  : 3e-05
  Max seq length : 128
  Label smoothing: 0.1
  Best metric    : chrF (primary for Twi)

  Test BLEU      : 27.30  (v1 baseline: 10.50)
  Test chrF      : 51.03  (v1 baseline: 32.30)
  Test TER       : 61.48  (v1 baseline: 87.64)

  Published      : https://huggingface.co/ninte/twi-en-nllb-v2
